# ModernBERT fine-tuning on Colab (issues #19 and #20)

This notebook trains ModernBERT-base to tag contract chunks with the 41 CUAD clause categories, on a
free Google GPU. The training code itself lives in `notebooks/modernbert_finetune.py`; this notebook
just sets up Colab and runs it. Plan: `docs/issue-19-modernbert-finetune-plan.md`.

**How to use it:** run the cells top to bottom, one at a time (click a cell, then press
**Shift + Enter**). Read the text above each code cell first; it says what the cell does and what
you should see.

**Before you start (one-time, on your laptop):**
1. In Google Drive, create a folder `cuad_finetune`, and inside it a folder `data`.
2. Upload `data/processed/chunks_df.parquet` and `data/processed/chunk_labels_df.parquet` from the
   repo into `cuad_finetune/data/`. (`data/processed/` is created by `main_pipeline.ipynb`.)

## 1. Turn on the GPU

A GPU does the matrix math in training many times faster than a laptop CPU. Colab gives you one
for free, but you have to ask for it:

**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.**

Then run the cell below. You should see a table naming a GPU (usually *Tesla T4*) and
`GPU available: True`. If it says `False`, the runtime type didn't change; try the menu again.

In [ ]:
!nvidia-smi
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Connect Google Drive

A Colab machine is temporary: when the session ends, everything on it is deleted. So anything worth
keeping (the trained model, the results, the input data) lives in your Google Drive, which this
cell attaches as a folder at `/content/drive/MyDrive`.

A pop-up will ask you to sign in and allow access; accept it. The cell then checks that the two
data files are where the "before you start" step put them.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/cuad_finetune")
for name in ["chunks_df.parquet", "chunk_labels_df.parquet"]:
    path = DRIVE_DIR / "data" / name
    print(("found    " if path.exists() else "MISSING  ") + str(path))

## 3. Get the code

This downloads ("clones") the project from GitHub onto the Colab machine. It uses the
`albert-finetune` branch, so **the code has to be pushed to GitHub first**; Colab can't see files
that only exist on your laptop.

Running it again later pulls the newest version of the branch instead of cloning twice. The last
line shows which commit you're on; it should match your latest push.

In [ ]:
import os

REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1N-contract-review-challenge.git"
BRANCH = "albert-finetune"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}/notebooks
!git log --oneline -1

## 4. Install the Python packages

Colab already has most of what we need (PyTorch, numpy, pandas, scikit-learn). This adds or
updates only the Hugging Face libraries.

We deliberately **don't** run `pip install -r requirements.txt` here. That file pins older numpy
and torch versions so the project works on Intel Macs; installing those on Colab would replace
its own versions and break other preinstalled packages. Takes about a minute.

In [ ]:
!pip install -q "transformers>=4.48" accelerate datasets
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

## 5. Copy the data into place

The training script reads `data/processed/` inside the repo, so this copies the two files from
Drive into it and checks their size. Expect **8,549 chunks** and **350,509 label rows**. The
script refuses to train on anything else, because different numbers mean the files come from an
old version of the pipeline.

In [ ]:
import shutil
import pandas as pd

processed = Path(REPO_DIR) / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
for name in ["chunks_df.parquet", "chunk_labels_df.parquet"]:
    shutil.copy(DRIVE_DIR / "data" / name, processed / name)

print("chunks:", len(pd.read_parquet(processed / "chunks_df.parquet")))
print("label rows:", len(pd.read_parquet(processed / "chunk_labels_df.parquet")))

## 6. Quick check before the real run (a few minutes)

This trains for just 20 steps and saves nothing. It answers two questions before you commit real
GPU time:

- **Does training work on this GPU?** The loss (the number the model is trying to shrink) must be
  a normal number. If the script stops with "loss became nan", re-run this cell with `--fp32`
  added; that turns off the fast half-precision mode.
- **How long will a full run take?** The last line estimates it from these 20 steps.

Lots of log text is normal. Warnings about `HF_TOKEN`, or about weights being "newly initialized",
are fine: the second one just means the 41-category output layer starts untrained, which is
exactly what fine-tuning then trains.

In [ ]:
!python modernbert_finetune.py --check

## 7. Train

Pick which run to do by setting `RUN_NAME` and `RUN_OPTIONS`, then run the cell:

| Run | `RUN_NAME` | `RUN_OPTIONS` | Issue |
|---|---|---|---|
| v1, plain (do this first) | `modernbert_v1` | `""` | #19 |
| v2, start at real base rates | `modernbert_v2_biasinit` | `"--bias-init"` | #20 |
| v3, class weighting | `modernbert_v3_posweight` | `"--pos-weight sqrt"` | #20 |
| v4, oversample rare (if time) | `modernbert_v4_oversample` | `"--oversample-rare 50"` | #20 |

**What you'll see:** a progress bar, the loss every 20 steps, and after each of the 3 epochs a line
of scores on the 66 tune contracts (`eval_micro_ap`, `eval_macro_ap_present`, ...). Higher AP is
better. After epoch 1 the script checks that micro AP is above 0.05 (the base rate is about 0.024)
and stops on its own if the model isn't learning.

**While it runs:** keep this browser tab open. Free Colab disconnects if the tab sits idle or is
closed, and it caps how long one session can last.

**If it disconnects:** re-run cells 1–5, then run this cell again with `--resume` added to
`RUN_OPTIONS`. It continues from the last finished epoch, which is saved in Drive.

Everything goes to `cuad_finetune/runs/<RUN_NAME>/` in your Drive: `model/` (about 600 MB) and
`results/` (scores and predicted probabilities).

In [ ]:
RUN_NAME = "modernbert_v1"
RUN_OPTIONS = ""

OUTPUT_DIR = DRIVE_DIR / "runs" / RUN_NAME
!python modernbert_finetune.py --run-name {RUN_NAME} {RUN_OPTIONS} --output-dir "{OUTPUT_DIR}"

## 8. Download the results

The scores and predicted probabilities belong in the repo, under `results/<RUN_NAME>/`, so
teammates working on #21 and #22 can use them. This zips the run's `results/` folder (a few MB; the
model stays in Drive) and downloads it to your laptop. Unzip it into a new folder
`results/<RUN_NAME>/` in the repo, so you end up with `results/<RUN_NAME>/metrics.json` and the
other files next to it.

In [ ]:
from google.colab import files

zip_path = shutil.make_archive(f"/content/{RUN_NAME}", "zip", root_dir=OUTPUT_DIR / "results")
print(open(OUTPUT_DIR / "results" / "metrics.json").read())
files.download(zip_path)